In [1]:
import numpy as np
import pandas as pd
from nltk.stem.snowball import SnowballStemmer
from nltk.corpus import stopwords
import re
import ftfy
import html
pd.set_option('display.max_colwidth', None)

In [2]:
df=pd.read_csv("development.csv",delimiter=",", index_col="Id")

### *Source* feature inspection

In [3]:
df['source'] = df['source'].replace('\\N', 'Unknown')
source_counts = df['source'].value_counts()

sources_for_ohe = source_counts[source_counts >= 100].index.to_list()
if 'Unknown' in sources_for_ohe: 
    sources_for_ohe.remove('Unknown')

sources_for_te = source_counts[(source_counts >= 5) & (source_counts < 100)].index.to_list()
if 'Unknown' in sources_for_te: 
    sources_for_te.remove('Unknown')

sources_other = source_counts[source_counts < 5].index.to_list()
if 'Unknown' not in sources_other:
    sources_other.append('Unknown')

df['source_log_count'] = df['source'].map(source_counts).fillna(0)
df.loc[df['source'] == 'Unknown', 'source_log_count'] = 0
df['source_log_count'] = np.log1p(df['source_log_count'])

df['source_te_raw'] = np.where(df['source'].isin(sources_for_te), df['source'], 'Other')

df['is_source_other'] = df['source'].isin(sources_other).astype(int)
df = df.drop(columns=['source'])

print(f"Features generated:")
print(f"1. OHE - Sources >= 100: {len(sources_for_ohe)} sources")
print(f"2. TE - Sources 5-99: {len(sources_for_te)} sources")
print(f"3. Other - Sources < 5 + Unknown: {len(sources_other)-1}")
print(f"4. Log Count (count=0 for unkown sources)")

Features generated:
1. OHE - Sources >= 100: 56 sources
2. TE - Sources 5-99: 432 sources
3. Other - Sources < 5 + Unknown: 870
4. Log Count (count=0 for unkown sources)


### *Title* feature inspection

In [ ]:
n_nan_title = df['title'].isna().sum()
n_empty_title = df['title'].astype(str).str.strip().eq('').sum()
n_placeholders_title = df['title'].astype(str).str.strip().eq('\\N').sum()
print(f"Number of NaN rows: {n_nan_title}")
print(f"Number of empty rows: {n_empty_title}")
print(f"Number of placeholders (\\N): {n_placeholders_title}")
print("Titles Sample:")
print(df['title'].sample(10))

### *Article* feature inspection

In [ ]:
n_nan_article = df['article'].isna().sum()
n_empty_article = df['article'].astype(str).str.strip().eq('').sum()
n_placeholders_article = df['article'].astype(str).str.strip().eq('\\N').sum()
print(f"Number of NaN rows: {n_nan_article}")
print(f"Number of empty rows: {n_empty_article}")
print(f"Number of placeholders (\\N): {n_placeholders_article}")
print("Articles Sample")
print(df['article'].sample(10))

### *PageRank* feature inspection

In [ ]:
n_nan_pr = df['page_rank'].isna().sum()
n_empty_pr = df['page_rank'].astype(str).str.strip().eq('').sum()
n_placeholders_pr = df['page_rank'].astype(str).str.strip().eq('\\N').sum()
rank_5=np.array([df['page_rank'].values==5]).sum()
print(f"Number of NaN rows: {n_nan_pr}")
print(f"Number of empty rows: {n_empty_pr}")
print(f"Number of placeholders (\\N): {n_placeholders_pr}")
print(f"Number of articles with PageRank 5: {rank_5}")
print(df['page_rank'].sample(10))

### *Timestamp* feature inspection 

In [ ]:
n_nan_time = df['timestamp'].isna().sum()
n_empty_time = df['timestamp'].astype(str).str.strip().eq('').sum()
n_placeholders_time = df['timestamp'].astype(str).str.strip().eq('\\N').sum()
n_uslesess_time=np.array([df['timestamp'].values=="0000-00-00 00:00:00"]).sum()
print(f"Number of NaN rows: {n_nan_time}")
print(f"Number of empty rows: {n_empty_time}")
print(f"Number of placeholders (\\N): {n_placeholders_time}")
print(f"Number invalid dates (0000-00-00 00:00:00): {n_uslesess_time}")
print(df['timestamp'].sample(10))

### *Timestamp* feature processing

In [ ]:
def process_timestamp(df):
    df = df.copy()
    
    df['dt_obj'] = pd.to_datetime(df['timestamp'], errors='coerce')
    df['has_date'] = df['dt_obj'].notna().astype(int)
    
    df['year'] = df['dt_obj'].dt.year.fillna(-1).astype(int)
    df['month'] = df['dt_obj'].dt.month.fillna(-1).astype(int)
    df['day_of_week'] = df['dt_obj'].dt.dayofweek.fillna(-1).astype(int)  
    df['hour'] = df['dt_obj'].dt.hour.fillna(-1).astype(int)
    
    df['quarter'] = df['dt_obj'].dt.quarter.fillna(-1).astype(int)
    
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
    
    is_weekday = df['day_of_week'].isin([0,1,2,3,4])
    is_working_hour = (df['hour'] >= 8) & (df['hour'] <= 18)
    df['is_business_hours'] = (is_weekday & is_working_hour).astype(int)

    
    labels = [0, 1, 2, 3] 
    df['time_of_day'] = pd.cut(df['hour'], bins=[-1, 5, 12, 18, 24], labels=labels, right=False)
    df['time_of_day'] = df['time_of_day'].cat.add_categories([-1]).fillna(-1).astype(int)
    df.loc[df['has_date'] == 0, 'time_of_day'] = -1
    df = df.drop(columns=['timestamp', 'dt_obj'])
    
    return df

df = process_timestamp(df)

timestamp_cols = ['has_date', 'year', 'month', 'day_of_week', 'time_of_day', 'is_weekend', 'is_business_hours', 'quarter']
print(f"Timestamp expanded columns: {timestamp_cols}")
print("Sample of 10 timestamps:")
print(df[timestamp_cols].sample(10))

### *Title* feature stemming

In [ ]:
from nltk.stem.snowball import SnowballStemmer
import re
import html
import pandas as pd
import ftfy

stemmer = SnowballStemmer("english")
stop_words = set(stopwords.words('english')) - {'us', 'it', 'am', 'pm'}

def process_title_final(df):
    df = df.copy()
    titles = df['title'].astype(str)
    
    df['title_has_percent'] = titles.str.contains(r'%').astype(int)
    df['title_has_currency'] = titles.str.contains(r'[$€£]').astype(int)
    df['title_has_score'] = titles.str.contains(r'\b\d+\s*-\s*\d+\b').astype(int)
    df['title_has_vs'] = titles.str.contains(r'\bvs\.?\b', case=False).astype(int)
    df['title_has_number'] = titles.str.contains(r'\b\d+\b').astype(int)
    df['title_has_questmark'] = titles.str.contains(r'\?').astype(int)
    df['title_has_colon'] = titles.str.contains(r':').astype(int)
    
    title_lens = titles.str.len().replace(0, 1)
    df['title_upper_ratio'] = titles.str.count(r'[A-Z]') / title_lens
    df['title_word_count'] = titles.str.split().str.len()


    def cleaner_smart(text):
        if pd.isna(text) or text == "": return ""
        text = str(text)
        
        text = html.unescape(text)
        text = ftfy.fix_text(text)
        text = re.sub(r"\bU\.S\.\b", "US", text)
        text = re.sub(r"\bU\.K\.\b", "UK", text)
        text = re.sub(r"\bE\.U\.\b", "EU", text)
        text = re.sub(r"\bU\.N\.\b", "UN", text)
        text = re.sub(r"\bU\.S\.A\.\b", "USA", text)
        text = re.sub(r'\b(US|EU|UN|UK|AI|NATO|USA|NASA)\b', lambda m: m.group(0).lower(), text)
        
        text = re.sub(r'http\S+', '', text)
        text = re.sub(r'<[^>]+>', ' ', text)
        
        text = re.sub(r'[^a-zA-Z0-9$€£%]', ' ', text)
        
        words = text.split()
        cleaned_words = []
        
        for w in words:

            if w.lower() in stop_words:
                continue


            if w.isdigit():
                cleaned_words.append(w)
                continue
                
            if w.isupper() and len(w) <= 4:
                cleaned_words.append(w.lower())
                continue
            
            cleaned_words.append(stemmer.stem(w.lower()))
            
        return " ".join(cleaned_words)

    df['title_clean'] = df['title'].apply(cleaner_smart)
    df = df.drop(columns=['title'])
    
    return df


df = process_title_final(df)

### *Article* feature stemming

In [ ]:
import re
import html
import numpy as np
import pandas as pd
import ftfy
import nltk
from nltk.stem.snowball import SnowballStemmer
from nltk.corpus import stopwords


KW_LISTS = {
    "kw_sport":  ["game", "team", "season", "player", "coach", "win", "score", "cup", "match", "league", "sport"],
    "kw_biz":    ["market", "price", "share", "stock", "invest", "profit", "trade", "economi", "billion", "bank"],
    "kw_tech":   ["softwar", "comput", "web", "internet", "googl", "appl", "microsoft", "device", "user", "tech"],
    "kw_health": ["cancer", "study", "drug", "patient", "diseas", "doctor", "health", "clinic", "medic", "virus"],
    "kw_ent":    ["movie", "film", "star", "music", "band", "album", "show", "tv", "actor", "award"],
}

def process_article_final_v6(df, kw_lists=KW_LISTS):
    df = df.copy()

    header_pattern = re.compile(r"^\s*(?!(?:UPDATE|BREAKING|EXCLUSIVE|BULLETIN)\b)[A-Z]{2,}(?:\s+[A-Z]{2,}){0,2}\s*(?:\([A-Z]+\))?\s*[-–]\s*")
    agency_pattern = re.compile(r"^\s*\(?(Reuters|AP|AFP|CNN|BBC|UPI)\)?\s*[-–]\s*", re.IGNORECASE)
    url_pattern = re.compile(r"http\S+")
    html_pattern = re.compile(r"<[^>]+>")
    
    sentence_split_pattern = re.compile(r'(?<=[.!?])\s+')

    df["article"] = df["article"].replace("\\N", "").fillna("").astype(str)
    df["article_len_log"] = np.log1p(df["article"].str.len()).astype(float)
    df["article_has_img"] = df["article"].str.contains(r"<img", case=False, regex=True).astype(int)

    def _base_clean(text):
        if not text: return ""
        text = html.unescape(text)
        text = ftfy.fix_text(text)
        
        text = re.sub(r"\bU\.S\.\b", "US", text)
        text = re.sub(r"\bU\.K\.\b", "UK", text)
        text = re.sub(r"\bE\.U\.\b", "EU", text)
        text = re.sub(r"\bU\.N\.\b", "UN", text)
        text = re.sub(r"\bU\.S\.A\.\b", "USA", text)
        
        text = url_pattern.sub(" ", text)
        text = html_pattern.sub(" ", text)
        
        lines = text.split("\n", 1)
        if lines:
            lines[0] = header_pattern.sub("", lines[0])
            lines[0] = agency_pattern.sub("", lines[0])
            text = "\n".join(lines)
        return text

    preclean_series = df["article"].map(lambda x: _base_clean(x).lower())

    for name, kws in kw_lists.items():
        pieces = [re.escape(k) + r"\w*" for k in kws]
        pattern = r"\b(?:" + "|".join(pieces) + r")\b"
        df[name] = preclean_series.str.count(pattern).astype(int)

    df["article_has_first_person"] = preclean_series.str.contains(r"\b(i|we|my|our|me)\b", regex=True).astype(int)
    df["article_has_quotes"] = preclean_series.str.contains(r'["“”]', regex=True).astype(int)
    df["article_has_money_or_pct"] = preclean_series.str.contains(r"[$€£%]", regex=True).astype(int)
    
    def _lede_clean(text):
        text = _base_clean(text) 
        if not text: 
            return ""
        sentences = sentence_split_pattern.split(text)
        lede_text = " ".join(sentences[:3])
        lede_text = re.sub(r"\b(US|UK|EU|UN|AI|NATO|USA|NASA|DARPA)\b", lambda m: m.group(0).lower(), lede_text)
        lede_text = re.sub(r"[^a-zA-Z0-9$€£%]", " ", lede_text)

        words = lede_text.split()
        if len(words) > 150:
            words = words[:150]

        cleaned = []
        for w in words:
            wl = w.lower()
            if wl in stop_words: continue
            if w.isdigit():
                cleaned.append(w)
                continue
            if w.isupper() and len(w) <= 5:
                cleaned.append(wl)
                continue
            cleaned.append(stemmer.stem(wl))
        
        return " ".join(cleaned)

    df["article_clean"] = df["article"].map(_lede_clean)    
    df = df.drop(columns=["article"])
    
    return df

In [ ]:
def inspect_post_processing(df, n_samples=10):
    cols_to_show = [
        "title_clean",
        "article_clean",
    ]

    sample_df = df[cols_to_show].sample(n=n_samples, random_state=42)
    for _, row in sample_df.iterrows():

        if "title_clean" in row:
            print("TITLE CLEAN:")
            print(row["title_clean"])
            print()

        if "article_clean" in row:
            print("ARTICLE CLEAN:")
            print(row["article_clean"])
            print()
        
        print("-"*50)

inspect_post_processing(df, n_samples=10)

### Encoding categorical features

In [ ]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from category_encoders import CatBoostEncoder
from sklearn.pipeline import Pipeline

def _to_1d(x):
    x = np.asarray(x)
    return x.ravel() if x.ndim > 1 else x

def make_source_ohe(x):
    s = _to_1d(x)
    s = np.where(np.isin(s, sources_for_ohe), s, "z_ignore")
    return s.reshape(-1, 1)

def make_is_other(x):
    s = _to_1d(x)
    return np.isin(s, sources_other).astype(int).reshape(-1, 1)

def make_source_te_raw(x):
    s = _to_1d(x)
    s = np.where(np.isin(s, sources_for_te), s, "Other")
    return s.reshape(-1, 1)

preprocessor = ColumnTransformer(
    transformers=[
        ("ohe", Pipeline([
            ("sel", FunctionTransformer(make_source_ohe, validate=False)),
            ("ohe", OneHotEncoder(
                categories=[sources_for_ohe + ["z_ignore"]],
                handle_unknown="ignore",
                drop=["z_ignore"],
                sparse_output=True
            ))
        ]), ["source"]),

        ("is_other", FunctionTransformer(make_is_other, validate=False), ["source"]),

        ("te", Pipeline([
            ("sel", FunctionTransformer(make_source_te_raw, validate=False)),
            ("te", CatBoostEncoder(cols=[0], a=10, sigma=0.05))
        ]), ["source"]),

        ("tfidf_title", TfidfVectorizer(max_features=30000, ngram_range=(1,2),
                                        sublinear_tf=True, stop_words=None), "title_clean"),

        ("tfidf_article", TfidfVectorizer(max_features=25000, ngram_range=(1,1),
                                          sublinear_tf=True, stop_words=None), "article_clean"),
    ],
    remainder="passthrough",
    verbose_feature_names_out=False
)
